# RQ1 보강 확인: 맥락만의 성질인가(A), 피험자 사이에 공유되는가(B), 무엇인가(C)
기준: 2026-10-05 논의. 코드는 `src/fmri/axes_k_curve.py`(A)와 `src/fmri/axes_shared.py`(B, C).

- **A 대조**: 같은 k 곡선을 사물 이름(obj_P1)과 원본 캡션(orig_P1)에도 돌려 맥락과 비교한다. 사물·원본도 12~20차원이면 충분하다면 저차원성은 맥락의 성질이 아니라 뇌 반응의 일반 성질이다.
- **B 공유**: 피험자 간 하위공간 겹침과 기능적 전이(a의 방향을 b에 적용).
- **C 해석**: 방향별로 점수 높은/낮은 이미지와 차원별 단어 연관.

Drive 준비는 `axes_k_curve.ipynb`와 같다(`embeddings_v2`, `colab_bundle`, 피험자별 fMRI 폴더).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fmri_capstone'
!rm -rf /content/repo && git clone -q https://github.com/geon-2/fmri-imagery-context /content/repo
!pip -q install scipy
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 axes_shared.py --selftest | tail -2

## A. 사물·원본 k 곡선 (맥락 곡선과 같은 설정, 부트스트랩만 200회로 줄임)
피험자 1은 두 모델, 나머지 피험자는 MPNet만. 한 번에 30분~1시간 이상 걸릴 수 있다.

In [ ]:
FM = {'subj01': f'{BASE}/fmri/roi_betas', 'subj02': f'{BASE}/fmri_subj02', 'subj05': f'{BASE}/fmri_subj05', 'subj07': f'{BASE}/fmri_subj07'}
for REP in ['obj_P1', 'orig_P1']:
    for SUBJ, MODELS in [('subj01', ['MPNet', 'CLIP']), ('subj02', ['MPNet']), ('subj05', ['MPNet']), ('subj07', ['MPNet'])]:
        for MODEL in MODELS:
            print('=====', REP, SUBJ, MODEL)
            !cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_k_curve.py \
              --fmri-dir {FM[SUBJ]} --emb-dir {BASE}/embeddings_v2 --model {MODEL} --rep {REP} \
              --out-dir {BASE}/axes --tag {SUBJ} --boot 200

### A 그림: 맥락 / 사물 / 원본의 RRR 곡선을 한 그림에

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharey=True)
for ax, (SUBJ, MODEL) in zip(axes, [('subj01', 'MPNet'), ('subj02', 'MPNet'), ('subj05', 'MPNet'), ('subj07', 'MPNet')]):
    for REP, col in [('ctx_P1', '#c44e52'), ('obj_P1', '#4c72b0'), ('orig_P1', '#55a868')]:
        try: R = json.load(open(f'{BASE}/axes/axes_{SUBJ}_{MODEL}_{REP}.json'))
        except Exception: continue
        c = R['curves']['RRR']; ax.plot([x['k'] for x in c], [x['P'] for x in c], 'o-', color=col, label=f'{REP} RRR')
        pc = R['curves']['PCA']; ax.plot([x['k'] for x in pc], [x['P'] for x in pc], 's--', color=col, alpha=.5, label=f'{REP} PCA')
        ax.axhline(R['full']['P'], color=col, lw=.7, ls=':')
        print(SUBJ, MODEL, REP, 'full', round(R['full']['P'], 3), 'k*', R['kstar']['RRR'], 'RRR k=5', round([x['P'] for x in c if x['k'] == 5][0], 3), 'k=12', round([x['P'] for x in c if x['k'] == 12][0], 3))
    ax.set_xscale('log'); ax.set_title(f'{SUBJ} {MODEL}'); ax.set_xlabel('k')
axes[0].set_ylabel('P'); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{BASE}/axes/A_compare_curves.png', dpi=150); plt.show()

## B와 C. 피험자 간 공유와 해석 (MPNet, ctx_P1)
하위공간 겹침, 반분할 기준, 기능적 전이(12개 방향 쌍)를 구하고, 피험자 1의 상위 12방향을 해석한다. 10~30분.

In [ ]:
SUBJECTS = ','.join(f'{k}:{v}' for k, v in FM.items())
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_shared.py \
  --subjects {SUBJECTS} --emb-dir {BASE}/embeddings_v2 --bundle-dir {BASE}/colab_bundle \
  --model MPNet --rep ctx_P1 --out-dir {BASE}/axes_shared --interpret subj01 --k 12

## CLIP으로도 (B, C 반복)

In [ ]:
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u axes_shared.py \
  --subjects {SUBJECTS} --emb-dir {BASE}/embeddings_v2 --bundle-dir {BASE}/colab_bundle \
  --model CLIP --rep ctx_P1 --out-dir {BASE}/axes_shared --interpret subj01 --k 12